# Impoliteness pilot — data prep

Design doc: `docs/superpowers/specs/2026-07-23-impoliteness-pilot-design.md`.

This section builds the paragraph-level dataset the pilot's sampling/LLM-scoring steps will
draw from: all paragraphs (regular speech + interjections) within a **±1-year window around
each state's own AfD entry date** — a per-state event window rather than one fixed calendar
year, so the pilot sample is anchored to the actual treatment timing per state instead of an
arbitrary shared year.

`DATA_ROOT` is set in two ways depending on environment:
- **Local**: read from `.env` in the project root (copy `.env.example` → `.env` and set your path)
- **Colab**: mount Google Drive in the cell below, then set `DATA_ROOT` to the Drive path


In [1]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = "/content/drive/My Drive/my_projects/M.A. Parliament/Code and Data/data"
    print("Status: Connected to Google Colab")
else:
    if "DATA_ROOT" not in os.environ:
        try:
            from dotenv import load_dotenv, find_dotenv
            load_dotenv(find_dotenv())
        except ImportError:
            pass
    DATA_ROOT = os.environ.get("DATA_ROOT", "")
    print("Status: Connected to a Local or Custom Kernel")

print("DATA_ROOT:", DATA_ROOT)
assert DATA_ROOT and os.path.isdir(DATA_ROOT), f"DATA_ROOT not set or missing: {DATA_ROOT!r}"


Status: Connected to a Local or Custom Kernel
DATA_ROOT: /Users/anna/Library/CloudStorage/GoogleDrive-anle.werner.01@gmail.com/My Drive/my_projects/M.A. Parliament/Code and Data/data


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW  = Path(DATA_ROOT) / "raw"
PROC = Path(DATA_ROOT) / "processed"
V3   = RAW / "stateparl_v3_parquet"


## Load paragraphs and the LLM-exploded nsc dataset

- `stateparl_v3_paragraphs.parquet` — every paragraph (speech + interjections), all affiliations.
- `nsc_llm.parquet` — output of `measurement/nsc_llm_explode.py`: one row per (nsc_type, party)
  atomic classification unit, exploded from `nsc.parquet`. Only covers `affiliation == "nsc"`
  paragraphs; a single paragraph can map to 0 (non-interjection row types, already excluded),
  1, or several `nsc_llm` rows (multi-segment / multi-type / multi-party source rows).


In [3]:
paragraphs = pd.read_parquet(V3 / "stateparl_v3_paragraphs.parquet")
paragraphs["date"] = pd.to_datetime(paragraphs["date"])
print(f"paragraphs: {paragraphs.shape[0]:,} rows x {paragraphs.shape[1]} cols, "
      f"{paragraphs['state'].nunique()} states")

nsc_llm = pd.read_parquet(PROC / "nsc_llm.parquet")
print(f"nsc_llm:    {nsc_llm.shape[0]:,} rows x {nsc_llm.shape[1]} cols, "
      f"{nsc_llm['paragraph_id'].nunique():,} distinct source paragraphs")


paragraphs: 16,078,467 rows x 14 cols, 16 states


nsc_llm:    6,176,081 rows x 26 cols, 4,112,535 distinct source paragraphs


## AfD entry date per state

First legislative period (per state) with any `affiliation == "afd"` row, using that period's
own constitutive-session date (first sitting overall, not just the first AfD-affiliated row) —
the same corpus-derived method validated in `analysis/nsc_analysis.ipynb`'s `AFD_PRESENCE`
derivation (0–29 day gap vs. the first actual AfD row, vs. up to ~11 months of error from the
old entry-year-only proxy). This pilot only needs the single first-entry "shock" date per
state, not the full presence/exit history `AFD_PRESENCE` tracks — Bremen and Schleswig-Holstein
later exited, but their *entry* window is still the relevant one for this pilot's ±1yr framing.


In [4]:
period_bounds = (
    paragraphs.groupby(["state", "period"])["date"]
    .min()
    .rename("period_start")
)

AFD_ENTRY = (
    paragraphs.loc[paragraphs["affiliation"] == "afd", ["state", "period"]]
    .drop_duplicates()
    .merge(period_bounds, on=["state", "period"])
    .sort_values(["state", "period"])
    .groupby("state")["period_start"]
    .first()
    .to_dict()
)

print(f"States with an AfD entry date: {len(AFD_ENTRY)} of {paragraphs['state'].nunique()}")
for state, entry in sorted(AFD_ENTRY.items()):
    print(f"  {state}: {entry.date()}")


States with an AfD entry date: 16 of 16
  bb: 2014-10-08
  be: 2016-10-27
  bw: 2016-05-11
  by: 2018-11-05
  hb: 2015-07-01
  he: 2019-01-18
  hh: 2015-09-16
  mv: 2016-10-04
  ni: 2017-11-14
  nw: 2017-06-27
  rp: 2016-05-18
  sh: 2017-06-06
  sl: 2017-04-25
  sn: 2014-09-29
  st: 2016-04-12
  th: 2014-10-14


## Filter to ±1 year around each state\'s AfD entry


In [5]:
WINDOW = pd.DateOffset(years=1)
ENTRY_WINDOW = {s: (d - WINDOW, d + WINDOW) for s, d in AFD_ENTRY.items()}

mask = pd.Series(False, index=paragraphs.index)
for state, (start, end) in ENTRY_WINDOW.items():
    mask |= (paragraphs["state"] == state) & paragraphs["date"].between(start, end)

para_window = paragraphs[mask].copy()
print(f"Filtered: {len(para_window):,} of {len(paragraphs):,} paragraphs "
      f"({len(para_window)/len(paragraphs)*100:.1f}%), {para_window['state'].nunique()} states")
print()
print(para_window.groupby("state").size().sort_values(ascending=False).to_string())


Filtered: 1,223,597 of 16,078,467 paragraphs (7.6%), 16 states

state
nw    112336
ni    107797
he     99050
mv     98940
bw     91776
th     87036
sh     82839
st     79069
by     72397
sn     67972
rp     63672
hh     62186
be     61660
hb     60729
bb     49326
sl     26812


## Join `nsc_llm` onto the filtered paragraphs

Left join on `paragraph_id` only. `nsc_llm` columns that already exist identically in
`paragraphs` (describable from `paragraph_id` alone, or a straight duplicate of an existing
column) are dropped first so the merge doesn\'t produce colliding/redundant columns:
`state`/`period`/`nth`/`date`/`protocol_position` are already in `paragraphs`; `protocol_id` and
`speech_id` likewise; `raw_row` duplicates `paragraphs`\' own `content` verbatim.

Non-`nsc` paragraphs (regular speech, government, president, ...) get exactly one output row
with all `nsc_llm` columns `NaN` — no match. `nsc`-affiliated paragraphs fan out to one row per
matching `nsc_llm` record (their nsc_type/party classification units), by design — a single
interjection paragraph can be one row (the common case) or several (multi-segment / multi-type /
multi-party source rows, see `nsc_llm_explode.py`).


In [6]:
_REDUNDANT = ["protocol_id", "speech_id", "state", "period", "nth", "date",
              "protocol_position", "raw_row"]
nsc_llm_slim = nsc_llm.drop(columns=_REDUNDANT)

merged = para_window.merge(nsc_llm_slim, on="paragraph_id", how="left")

print(f"para_window: {len(para_window):,} rows  ->  merged: {len(merged):,} rows")
print(f"  (+{len(merged) - len(para_window):,} from nsc paragraphs fanning out to multiple "
      f"classification-unit rows)")
print()

n_nsc_paragraphs = (para_window["affiliation"] == "nsc").sum()
n_matched_paragraphs = merged.loc[merged["nsc_type"].notna(), "paragraph_id"].nunique()
print(f"nsc-affiliated paragraphs in window: {n_nsc_paragraphs:,}")
print(f"  of which matched >=1 nsc_llm row:  {n_matched_paragraphs:,}")
print(f"  (gap = non-interjection nsc_type rows, e.g. glocke/Prozedural/mislabelled/noise/"
      f"garbled — already excluded from nsc_llm by design)")
print()

non_nsc = merged[merged["affiliation"] != "nsc"]
print(f"Non-nsc rows: {len(non_nsc):,}; any duplicated paragraph_id (should be False)? "
      f"{non_nsc['paragraph_id'].duplicated().any()}")
print(f"Non-nsc rows with non-null nsc_type (should be 0): {non_nsc['nsc_type'].notna().sum()}")


para_window: 1,223,597 rows  ->  merged: 1,399,244 rows
  (+175,647 from nsc paragraphs fanning out to multiple classification-unit rows)

nsc-affiliated paragraphs in window: 324,761
  of which matched >=1 nsc_llm row:  317,934
  (gap = non-interjection nsc_type rows, e.g. glocke/Prozedural/mislabelled/noise/garbled — already excluded from nsc_llm by design)

Non-nsc rows: 898,836; any duplicated paragraph_id (should be False)? False
Non-nsc rows with non-null nsc_type (should be 0): 0


## Sanity check: preview a joined nsc row and a joined non-nsc row


In [7]:
cols = ["paragraph_id", "state", "period", "date", "affiliation", "content",
        "nsc_type", "party_canonical", "affiliation_derived", "speaker_name", "content_text"]

print("-- example nsc paragraph (fanned out) --")
example_pid = merged.loc[merged["nsc_type"].notna(), "paragraph_id"].iloc[0]
print(merged[merged["paragraph_id"] == example_pid][cols].to_string(index=False))

print()
print("-- example non-nsc paragraph (single row, nsc columns NaN) --")
print(merged[merged["affiliation"] != "nsc"][cols].head(2).to_string(index=False))


-- example nsc paragraph (fanned out) --
 paragraph_id state  period       date affiliation                   content nsc_type party_canonical affiliation_derived speaker_name content_text
       306811    bb       5 2013-11-20         nsc (Ha, ha! bei CDU und FDP)    Zuruf             CDU                 cdu                   Ha, ha!
       306811    bb       5 2013-11-20         nsc (Ha, ha! bei CDU und FDP)    Zuruf             FDP                 fdp                   Ha, ha!

-- example non-nsc paragraph (single row, nsc columns NaN) --


 paragraph_id state  period       date affiliation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                content nsc_type party_canonical affiliation_derived speaker_name content_text
       306809    bb       5 2013-11-20         fdp Herr Präsident! Die FDP-Fraktion beantragt die Absetzung des Tagesordnungspunktes 6 von der Tagesordnung „Gesetz zur Änderung des Gesetzes über die Feststellung des Haushaltsplanes des Landes Brandenburg". Die Sondersitzung des 

## LLM scoring — zero-shot impoliteness classification

Sample paragraphs from `merged` and classify each with a local Qwen3-14B model via Ollama.
See `docs/superpowers/specs/2026-07-23-impoliteness-pilot-design.md` (Part 2) for the design
rationale: binary output, one call per paragraph, everything seeded.

In [8]:
import subprocess

MODEL = "qwen3:14b-q8_0"

result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
if MODEL not in result.stdout:
    print(f"Pulling {MODEL} (this downloads ~30GB, may take a while)...")
    subprocess.run(["ollama", "pull", MODEL], check=True)
else:
    print(f"{MODEL} already pulled.")

qwen3:14b-q8_0 already pulled.


In [9]:
SEED = 42
N_SAMPLE = 300

merged["text_to_classify"] = merged["content_text"].where(
    merged["content_text"].notna(), merged["content"]
)

# One row per physical paragraph/segment, not per (nsc_type, party) classification
# unit - otherwise a multi-party interjection (same text, several nsc_llm rows)
# would get classified and counted multiple times.
dedup = merged.drop_duplicates(subset=["paragraph_id", "segment_idx"], keep="first")
sample = dedup.sample(n=N_SAMPLE, random_state=SEED).reset_index(drop=True)

print(f"Deduplicated {len(merged):,} rows -> {len(dedup):,} unique paragraph/segment texts")
print(f"Sampled {len(sample):,} rows for LLM scoring")
sample[["paragraph_id", "state", "period", "affiliation", "text_to_classify"]].head()

Deduplicated 1,399,244 rows -> 1,277,020 unique paragraph/segment texts
Sampled 300 rows for LLM scoring


,paragraph_id,state,period,affiliation,text_to_classify
0,11544705,rp,17,nsc,Beifall
1,3609659,by,18,afd,"Es geht hier eben nicht darum, dass es Frauen ..."
2,10618766,nw,17,afd,In Ihrem Antrag sprechen Sie doch selbst die G...
3,1269053,be,17,pre,Dringliche Beschlussempfehlung des Ausschusses...
4,15559405,th,5,cdu,"Der federführ ende Ausschuss für Soziales, Fam..."


In [10]:
import ollama

from impoliteness_lib import build_prompt, parse_response

_test_response = ollama.chat(
    model=MODEL,
    messages=build_prompt("Das ist doch eine Frechheit, Sie Lügner!"),
    think=False,
    format="json",
    options={"temperature": 0, "seed": SEED},
)
print(_test_response["message"]["content"])
print(parse_response(_test_response["message"]["content"]))

{"impolite": true, "reason": "Verwendung von Beleidigungen und Anschuldigungen"}
{'impolite': True, 'reason': 'Verwendung von Beleidigungen und Anschuldigungen', 'raw_output': '{"impolite": true, "reason": "Verwendung von Beleidigungen und Anschuldigungen"}'}
